In [33]:
import pandas as pd
import numpy as np

#Load messy dataset
df = pd.read_csv('marketing_campaign_data_messy.csv')

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 2020 rows, 12 columns


In [34]:
df.head(3)

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM


# Header cleaning

In [35]:
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']


In [36]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
df.head()

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,clicks,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


# Type conversion & currency cleaning

In [37]:
dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask ,['campaign_id','spend']].head(3))

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22


In [38]:
df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]','', regex = True)
df['spend'] = pd.to_numeric(df['spend'])

print(df.loc[dirty_spend_mask ,['campaign_id','spend']].head(3))

   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


# Categorical Typos

In [39]:
print(df['channel'].unique())
cleanup_map = {
    'Facebok':'Facebook',
    'Insta_gram':'Instagram',
    'Tik_Tok':'TikTok',
    'E-mail' : 'Email',
    'Gogle':'Google Ads' ,
    'N/A': np.nan
}

df['channel'] = df['channel'].replace(cleanup_map)
print(df['channel'].unique())


['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


# Handling mixed booleans

In [40]:
print(df['active'].unique())
bool_map = {
'Yes': True,
'True': True,
'Y': True,
'1': True,
'No': False,
'False': False,
'0': False,
'N': False
}

df['active']= df['active'].map(bool_map).fillna(False).astype(bool)
print(df['active'].unique())

['Y' '0' 'No' 'True' 'Yes' '1' 'False']
[ True False]


# Date parsing

In [41]:
print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'],format='mixed',dayfirst= True, errors = 'coerce')
df['end_date'] = pd.to_datetime(df['end_date'],format='mixed',dayfirst= True, errors = 'coerce')

print(df['start_date'].dtype)

object
datetime64[ns]


# Logical Integrity (clicks vs impressions)

In [42]:
df = df.loc[:,~df.columns.duplicated()]
impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask ,['campaign_id','impressions','clicks']].head(3))


Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


# Logical integrity (Time travel)

In [43]:
time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask, ['campaign_id','start_date','end_date']].head(3))

# It will be assumed that the duration of the campaign is 30 days, hence the following approach to this solution for the errors that were found
df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask,'start_date'] + pd.Timedelta(days=30)
print(df.loc[time_travel_mask, ['campaign_id','start_date','end_date']].head(3))


   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


# Handling outliers (winsorizing)

In [44]:
Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1 
upper_limit = round(Q3 + (3 * IQR),2)

outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask, ['campaign_id','spend']].head(3))

df.loc[outlier_mask,'spend'] = upper_limit
print(df.loc[outlier_mask, ['campaign_id','spend']].head(3))

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
     campaign_id    spend
789    CMP-00790  8603.54
1443   CMP-01444  8603.54
1460   CMP-01461  8603.54


# String parsing (feature extraction)

In [45]:
print(df['campaign_name'].head(3))
df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')
print(df[['campaign_name','season']].head(3))


0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter
